# Pratica — Lezione 11. Architetture di agenti: da A3C a IMPALA

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

In classe hai visto "Catch": il banco di prova consigliato prima di complicare qualsiasi agente con il parallelismo (A3C) o con architetture distribuite (IMPALA). In questa pratica lavori con lo stesso ambiente in due esercizi:

1. **Completi tu stesso la funzione di passo (`step`) dell'ambiente**, da zero: come cade la palla e come si muove la racchetta
2. **Parametrizzi la dimensione della griglia e la velocità di caduta**, e confronti quanti episodi serve all'agente per imparare bene in ciascuna configurazione

Ti bastano solo `numpy`, `torch` e `matplotlib`, tutte e tre preinstallate su Google Colab.

**Prima di iniziare, esegui la cella seguente per importare le librerie e fissare il seed.**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0)
torch.manual_seed(0)


## Esercizio 1 — Completa la funzione di passo di "Catch"

In classe hai visto la classe `Catch` già costruita. Qui hai la stessa struttura, ma con il metodo `step` vuoto: tocca a te completarlo.

Ricorda la meccanica del gioco: la palla compare in una colonna casuale della riga superiore e scende di una riga per passo; controlli una racchetta larga `ANCHO_PALA` pixel nella riga inferiore, che si muove a sinistra, a destra o resta ferma. Quando la palla arriva alla riga inferiore, ricevi +1 se la racchetta la copre e -1 se non la copre.

**Il tuo compito è completare `step`, dove c'è `# TODO`:**

1. Muovi la racchetta in base all'azione ricevuta (0 = sinistra, 1 = ferma, 2 = destra), senza permetterle di uscire dalla griglia
2. Fai cadere la palla di una riga
3. Determina se l'episodio termina (la palla è arrivata all'ultima riga)
4. Se è terminato, calcola la ricompensa: +1.0 se la racchetta copre la colonna della palla, -1.0 se no. Se non è terminato, la ricompensa è 0.0
5. Restituisci il nuovo stato, la ricompensa e se è terminato

**Suggerimenti**

- Il dizionario `{0: -1, 1: 0, 2: 1}` traduce ogni azione in uno spostamento di colonna
- Usa `np.clip(valore, minimo, massimo)` per impedire che la racchetta esca dalla griglia: l'intervallo valido di `columna_pala` va da 0 a `COLUMNAS - ANCHO_PALA`
- L'episodio termina quando `self.fila_pelota == FILAS - 1`
- La racchetta copre la palla quando `self.columna_pala <= self.columna_pelota < self.columna_pala + ANCHO_PALA`
- Lo stato si calcola sempre allo stesso modo, ed è già implementato in `_estado()`: non serve toccarlo

Con il `# TODO` non completato, la cella di prova qui sotto fallisce: è il segnale che ti manca ancora da terminare `step`.

In [ ]:
# --- Andamiaje del entorno "Catch": completa step() ---
FILAS = 10
COLUMNAS = 5
ANCHO_PALA = 2
N_ACCIONES = 3  # 0 = izquierda, 1 = quieta, 2 = derecha


class Catch:
    """Implementacion minima del juego de juguete "Catch"."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.fila_pelota = 0
        self.columna_pelota = np.random.randint(COLUMNAS)
        # Punto de partida aleatorio de la pala, sin salirse del borde
        self.columna_pala = np.random.randint(COLUMNAS - ANCHO_PALA + 1)
        return self._estado()

    def _estado(self):
        """Codifica el estado como una imagen binaria aplanada: un uno
        donde esta la pelota y unos donde ocupa la pala. Ya esta completa,
        no hace falta que la toques."""
        grid = np.zeros((FILAS, COLUMNAS), dtype=np.float32)
        grid[self.fila_pelota, self.columna_pelota] = 1.0
        grid[FILAS - 1, self.columna_pala:self.columna_pala + ANCHO_PALA] = 1.0
        return grid.flatten()

    def step(self, accion):
        # TODO 1: mueve la pala segun `accion` (0=izquierda, 1=quieta, 2=derecha)
        #         usa np.clip para no salirte del rango [0, COLUMNAS - ANCHO_PALA]

        # TODO 2: haz que la pelota caiga una fila (self.fila_pelota)

        # TODO 3: determina si el episodio termina: la pelota llego a la
        #         ultima fila (FILAS - 1)

        # TODO 4: si termino, calcula la recompensa (+1.0 si la pala cubre
        #         la columna de la pelota, -1.0 si no). Si no termino,
        #         la recompensa es 0.0

        # TODO 5: devuelve self._estado(), la recompensa y si termino
        pass


**Esegui la cella seguente per provare la tua implementazione.** Posiziona la palla e la racchetta in posizioni note e verifica che `step` si comporti come ti aspetti: che la racchetta si muova, che la palla cada, che la ricompensa compaia solo alla fine e che colpisca o manchi in base al caso.

In [ ]:
# --- Celda de prueba: no hace falta que la edites ---
entorno_prueba = Catch()

# Paso intermedio: la pelota todavia no llega abajo
entorno_prueba.fila_pelota = 0
entorno_prueba.columna_pelota = 2
entorno_prueba.columna_pala = 0
_, recompensa, terminado = entorno_prueba.step(1)  # 1 = quieta
assert entorno_prueba.fila_pelota == 1, "la pelota deberia caer una fila"
assert entorno_prueba.columna_pala == 0, "con accion 'quieta' la pala no deberia moverse"
assert recompensa == 0.0 and not terminado, "a mitad de la caida no hay recompensa"
print("Paso intermedio: OK")

# Ultimo paso, pala mal colocada: debe fallar
entorno_prueba.fila_pelota = FILAS - 2
entorno_prueba.columna_pelota = 4
entorno_prueba.columna_pala = 0
_, recompensa, terminado = entorno_prueba.step(1)
assert terminado, "en la ultima fila el episodio debe terminar"
assert recompensa == -1.0, "la pala no cubre la pelota: deberia fallar"
print("Fallo esperado: OK")

# Ultimo paso, pala bien colocada: debe atrapar
entorno_prueba.fila_pelota = FILAS - 2
entorno_prueba.columna_pelota = 4
entorno_prueba.columna_pala = 3
_, recompensa, terminado = entorno_prueba.step(1)
assert terminado and recompensa == 1.0, "la pala cubre la pelota: deberia atrapar"
print("Acierto esperado: OK")

# La pala no debe salirse de los bordes
entorno_prueba.columna_pala = 0
entorno_prueba.fila_pelota = 0
entorno_prueba.step(0)  # izquierda, ya en el borde
assert entorno_prueba.columna_pala == 0, "no deberia salirse del borde izquierdo"
entorno_prueba.columna_pala = COLUMNAS - ANCHO_PALA
entorno_prueba.step(2)  # derecha, ya en el borde
assert entorno_prueba.columna_pala == COLUMNAS - ANCHO_PALA, "no deberia salirse del borde derecho"
print("Bordes: OK")

print("\n¡Los cuatro controles pasan! Tu step() funciona correctamente.")


**Con `step` funzionante, addestra un agente vero per confermarlo.** Riutilizziamo lo stesso actor-critic della lezione, con meno episodi così vedi il risultato in fretta.

**Esegui questa cella: se il tuo ambiente funziona, il tasso di successi degli ultimi 100 episodi dovrebbe risultare molto più alto di quello dei primi 100.**

In [ ]:
# --- Entrenamiento corto para confirmar que el entorno funciona de punta a punta ---
class ActorCritico(nn.Module):
    def __init__(self, dim_entrada, n_acciones, dim_oculta=128):
        super().__init__()
        self.base = nn.Sequential(nn.Linear(dim_entrada, dim_oculta), nn.ReLU())
        self.cabeza_politica = nn.Linear(dim_oculta, n_acciones)
        self.cabeza_valor = nn.Linear(dim_oculta, 1)

    def forward(self, x):
        h = self.base(x)
        return self.cabeza_politica(h), self.cabeza_valor(h)


GAMMA = 0.99
TASA_APRENDIZAJE = 1e-3
COEF_VALOR = 0.5
COEF_ENTROPIA = 0.01
N_EPISODIOS_PRUEBA = 1500

torch.manual_seed(0)
np.random.seed(0)
entorno = Catch()
red = ActorCritico(dim_entrada=FILAS * COLUMNAS, n_acciones=N_ACCIONES)
optimizador = torch.optim.Adam(red.parameters(), lr=TASA_APRENDIZAJE)


def ejecutar_episodio():
    estado = entorno.reset()
    log_probs, valores, recompensas, entropias = [], [], [], []
    terminado = False
    while not terminado:
        entrada = torch.from_numpy(estado)
        logits, valor = red(entrada)
        distribucion = torch.distributions.Categorical(logits=logits)
        accion = distribucion.sample()
        estado, recompensa, terminado = entorno.step(accion.item())
        log_probs.append(distribucion.log_prob(accion))
        entropias.append(distribucion.entropy())
        valores.append(valor.squeeze())
        recompensas.append(recompensa)
    return log_probs, valores, recompensas, entropias


def actualizar_red(log_probs, valores, recompensas, entropias):
    retornos = []
    G = 0.0
    for r in reversed(recompensas):
        G = r + GAMMA * G
        retornos.insert(0, G)
    retornos = torch.tensor(retornos, dtype=torch.float32)
    valores = torch.stack(valores)
    ventaja = retornos - valores.detach()
    perdida_actor = -(torch.stack(log_probs) * ventaja).mean()
    perdida_critico = F.mse_loss(valores, retornos)
    perdida_entropia = torch.stack(entropias).mean()
    perdida = perdida_actor + COEF_VALOR * perdida_critico - COEF_ENTROPIA * perdida_entropia
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


aciertos = []
for episodio in range(N_EPISODIOS_PRUEBA):
    log_probs, valores, recompensas, entropias = ejecutar_episodio()
    actualizar_red(log_probs, valores, recompensas, entropias)
    aciertos.append(1 if recompensas[-1] > 0 else 0)

print("Tasa de aciertos, primeros 100 episodios:", round(np.mean(aciertos[:100]), 2))
print("Tasa de aciertos, ultimos 100 episodios:", round(np.mean(aciertos[-100:]), 2))


<details>
<summary>💡 Vedi la soluzione</summary>

```python
class Catch:
    """Implementazione minima del gioco giocattolo "Catch"."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.fila_pelota = 0
        self.columna_pelota = np.random.randint(COLUMNAS)
        self.columna_pala = np.random.randint(COLUMNAS - ANCHO_PALA + 1)
        return self._estado()

    def _estado(self):
        grid = np.zeros((FILAS, COLUMNAS), dtype=np.float32)
        grid[self.fila_pelota, self.columna_pelota] = 1.0
        grid[FILAS - 1, self.columna_pala:self.columna_pala + ANCHO_PALA] = 1.0
        return grid.flatten()

    def step(self, accion):
        # 1. Muoviamo la racchetta; np.clip evita che esca dalla griglia
        desplazamiento = {0: -1, 1: 0, 2: 1}[accion]
        self.columna_pala = int(
            np.clip(self.columna_pala + desplazamiento, 0, COLUMNAS - ANCHO_PALA)
        )
        # 2. La palla cade di una riga a ogni passo
        self.fila_pelota += 1
        # 3. L'episodio termina quando la palla arriva all'ultima riga
        terminado = self.fila_pelota == FILAS - 1
        # 4. C'è una ricompensa diversa da zero solo alla fine
        if terminado:
            atrapada = (
                self.columna_pala
                <= self.columna_pelota
                < self.columna_pala + ANCHO_PALA
            )
            recompensa = 1.0 if atrapada else -1.0
        else:
            recompensa = 0.0
        # 5. Restituiamo il nuovo stato, la ricompensa e se è terminato
        return self._estado(), recompensa, terminado
```
</details>

## Esercizio 2 — Parametrizza la difficoltà e confronta quanto tempo l'agente impiega a imparare

In classe hai visto che "Catch" usa costanti fisse: dieci righe, cinque colonne, una racchetta di due pixel, una palla che cade di una riga per passo. Ora trasformerai questi valori in parametri configurabili, per poter confrontare configurazioni diverse senza toccare ogni volta il codice dell'ambiente.

**Il tuo compito ha due parti, contrassegnate con `# TODO` nella cella seguente:**

**Parte A — Parametrizza l'ambiente.** Completa `CatchParametrizable` in modo che `filas`, `columnas`, `ancho_pala` e `velocidad_caida` (di quante righe cade la palla a ogni passo) siano attributi configurabili dal costruttore, invece delle costanti globali `FILAS`, `COLUMNAS` e `ANCHO_PALA` usate nell'esercizio uno.

**Parte B — Progetta l'esperimento.** Completa il ciclo che addestra un agente con ciascuna configurazione di `EXPERIMENTOS` e salva, per ognuna, quanti episodi ha impiegato per raggiungere un tasso di successi di 0.8 (calcolato con una media mobile di 100 episodi).

**Suggerimenti**

- Nel costruttore, salva ogni argomento come attributo d'istanza: `self.filas = filas`, e lo stesso per gli altri tre
- Dentro `reset`, `_estado` e `step`, sostituisci ogni riferimento a `FILAS`, `COLUMNAS` e `ANCHO_PALA` con `self.filas`, `self.columnas` e `self.ancho_pala`
- La palla ora cade di `self.velocidad_caida` righe per passo, non di una sola; l'episodio termina quando `self.fila_pelota >= self.filas - 1`
- Per la parte B, usa la funzione `entrenar_agente(entorno, n_episodios)` che trovi già pronta più sotto: restituisce la lista dei successi (uno o zero) episodio per episodio
- Anche `episodios_hasta_umbral(aciertos)` è già pronta: restituisce l'episodio in cui la media mobile ha raggiunto per la prima volta 0.8, oppure `None` se non ci arriva mai
- `EXPERIMENTOS` è un dizionario da nome a argomenti dell'ambiente; scorrilo con `.items()`

In [ ]:
# --- Parte A: completa la parametrizacion del entorno ---
N_ACCIONES = 3  # 0 = izquierda, 1 = quieta, 2 = derecha


class CatchParametrizable:
    """Version de "Catch" con tamano de cuadricula, ancho de pala y
    velocidad de caida configurables."""

    def __init__(self, filas=10, columnas=5, ancho_pala=2, velocidad_caida=1):
        # TODO: guarda los cuatro argumentos como atributos de instancia
        # (self.filas, self.columnas, self.ancho_pala, self.velocidad_caida)
        self.reset()

    def reset(self):
        self.fila_pelota = 0
        # TODO: usa self.columnas y self.ancho_pala en vez de las
        # constantes globales para elegir columna_pelota y columna_pala
        return self._estado()

    def _estado(self):
        # TODO: construye la grid con np.zeros usando self.filas y
        # self.columnas, marca la posicion de la pelota y la de la pala
        # (igual que en el ejercicio uno, pero con los atributos de instancia)
        pass

    def step(self, accion):
        # TODO: mueve la pala con self.columnas y self.ancho_pala
        # TODO: haz que la pelota caiga self.velocidad_caida filas
        # TODO: el episodio termina cuando self.fila_pelota >= self.filas - 1
        # TODO: calcula la recompensa igual que en el ejercicio uno
        # TODO: devuelve self._estado(), la recompensa y si termino
        pass


**Esegui la cella seguente per verificare la parte A.** Crea due ambienti di dimensioni diverse e controlla che lo stato che restituiscono abbia la dimensione corretta in ciascun caso.

In [ ]:
# --- Celda de prueba de la parte A: no hace falta que la edites ---
entorno_5x10 = CatchParametrizable(filas=10, columnas=5, ancho_pala=2, velocidad_caida=1)
entorno_8x10 = CatchParametrizable(filas=10, columnas=8, ancho_pala=2, velocidad_caida=1)

estado_5x10 = entorno_5x10.reset()
estado_8x10 = entorno_8x10.reset()

assert estado_5x10.shape[0] == 10 * 5, "el estado deberia tener 10*5 posiciones"
assert estado_8x10.shape[0] == 10 * 8, "el estado deberia tener 10*8 posiciones"
print("Tamano de estado por configuracion: OK")

estado, recompensa, terminado = entorno_5x10.step(1)
assert not terminado, "con velocidad_caida=1 en fila 0 no deberia terminar todavia"
print("Un paso de step(): OK")

print("\n¡La parametrizacion funciona! Sigue con la parte B.")


In [ ]:
# --- Actor-critico y funciones de entrenamiento reutilizables (ya completas) ---
GAMMA = 0.99
TASA_APRENDIZAJE = 1e-3
COEF_VALOR = 0.5
COEF_ENTROPIA = 0.01


class ActorCritico(nn.Module):
    def __init__(self, dim_entrada, n_acciones, dim_oculta=128):
        super().__init__()
        self.base = nn.Sequential(nn.Linear(dim_entrada, dim_oculta), nn.ReLU())
        self.cabeza_politica = nn.Linear(dim_oculta, n_acciones)
        self.cabeza_valor = nn.Linear(dim_oculta, 1)

    def forward(self, x):
        h = self.base(x)
        return self.cabeza_politica(h), self.cabeza_valor(h)


def ejecutar_episodio(entorno, red):
    estado = entorno.reset()
    log_probs, valores, recompensas, entropias = [], [], [], []
    terminado = False
    while not terminado:
        entrada = torch.from_numpy(estado)
        logits, valor = red(entrada)
        distribucion = torch.distributions.Categorical(logits=logits)
        accion = distribucion.sample()
        estado, recompensa, terminado = entorno.step(accion.item())
        log_probs.append(distribucion.log_prob(accion))
        entropias.append(distribucion.entropy())
        valores.append(valor.squeeze())
        recompensas.append(recompensa)
    return log_probs, valores, recompensas, entropias


def actualizar_red(red, optimizador, log_probs, valores, recompensas, entropias):
    retornos = []
    G = 0.0
    for r in reversed(recompensas):
        G = r + GAMMA * G
        retornos.insert(0, G)
    retornos = torch.tensor(retornos, dtype=torch.float32)
    valores = torch.stack(valores)
    ventaja = retornos - valores.detach()
    perdida_actor = -(torch.stack(log_probs) * ventaja).mean()
    perdida_critico = F.mse_loss(valores, retornos)
    perdida_entropia = torch.stack(entropias).mean()
    perdida = perdida_actor + COEF_VALOR * perdida_critico - COEF_ENTROPIA * perdida_entropia
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


def entrenar_agente(entorno, n_episodios, semilla=0):
    """Entrena un actor-critico desde cero sobre `entorno` y devuelve la
    lista de aciertos (1 o 0) episodio a episodio."""
    torch.manual_seed(semilla)
    np.random.seed(semilla)
    dim_entrada = entorno.filas * entorno.columnas
    red = ActorCritico(dim_entrada=dim_entrada, n_acciones=N_ACCIONES)
    optimizador = torch.optim.Adam(red.parameters(), lr=TASA_APRENDIZAJE)
    aciertos = []
    for _ in range(n_episodios):
        log_probs, valores, recompensas, entropias = ejecutar_episodio(entorno, red)
        actualizar_red(red, optimizador, log_probs, valores, recompensas, entropias)
        aciertos.append(1 if recompensas[-1] > 0 else 0)
    return aciertos


def episodios_hasta_umbral(aciertos, ventana=100, umbral=0.8):
    """Devuelve el primer episodio en el que la media movil de aciertos
    alcanza `umbral`, o None si nunca lo alcanza."""
    if len(aciertos) < ventana:
        return None
    tasa_movil = np.convolve(aciertos, np.ones(ventana) / ventana, mode="valid")
    indices = np.where(tasa_movil >= umbral)[0]
    if len(indices) == 0:
        return None
    return int(indices[0] + ventana)


**Parte B.** Completa il ciclo sperimentale della cella seguente: scorri `EXPERIMENTOS`, addestra un agente con ogni configurazione e salva in `resultados` quanti episodi ha impiegato per raggiungere l'80% di successi. Hai già due configurazioni di esempio in `EXPERIMENTOS`; puoi aggiungere le tue cambiando `columnas`, `ancho_pala` o `velocidad_caida`.

**Avviso:** questa cella addestra più agenti in sequenza e può richiedere uno o due minuti su Colab.

In [ ]:
# --- Parte B: completa el bucle experimental ---
N_EPISODIOS = 2000

EXPERIMENTOS = {
    "base (10x5, pala=2, vel=1)": dict(filas=10, columnas=5, ancho_pala=2, velocidad_caida=1),
    "mas columnas (10x8, pala=2, vel=1)": dict(filas=10, columnas=8, ancho_pala=2, velocidad_caida=1),
    # TODO: anade aqui una tercera configuracion propia, por ejemplo
    # cambiando ancho_pala a 1 o velocidad_caida a 2
}

resultados = {}

# TODO: recorre EXPERIMENTOS con .items(), crea un CatchParametrizable con
# cada configuracion (usa **argumentos), entrena con entrenar_agente y
# guarda en resultados[nombre] una tupla (aciertos, episodios_hasta_umbral)


In [ ]:
# --- Graficamos la comparacion (ya completa, no hace falta tocarla) ---
plt.figure(figsize=(9, 5))
for nombre, (aciertos, ep_umbral) in resultados.items():
    ventana = 100
    tasa_movil = np.convolve(aciertos, np.ones(ventana) / ventana, mode="valid")
    etiqueta = f"{nombre} — 80% en episodio {ep_umbral}" if ep_umbral else f"{nombre} — no llega al 80%"
    plt.plot(tasa_movil, label=etiqueta)

plt.xlabel("Episodios")
plt.ylabel("Tasa de aciertos (media movil de 100 episodios)")
plt.title('Efecto del tamano de cuadricula y la velocidad de caida en "Catch"')
plt.axhline(0.8, color="gray", linestyle="--", alpha=0.5)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for nombre, (_, ep_umbral) in resultados.items():
    print(nombre, "-> episodios hasta el 80% de aciertos:", ep_umbral)


<details>
<summary>💡 Vedi la soluzione</summary>

```python
# --- Parte A: ambiente parametrizzato ---
class CatchParametrizable:
    # Versione di "Catch" con dimensione della griglia, larghezza della racchetta e
    # velocità di caduta configurabili.

    def __init__(self, filas=10, columnas=5, ancho_pala=2, velocidad_caida=1):
        self.filas = filas
        self.columnas = columnas
        self.ancho_pala = ancho_pala
        self.velocidad_caida = velocidad_caida
        self.reset()

    def reset(self):
        self.fila_pelota = 0
        self.columna_pelota = np.random.randint(self.columnas)
        self.columna_pala = np.random.randint(self.columnas - self.ancho_pala + 1)
        return self._estado()

    def _estado(self):
        grid = np.zeros((self.filas, self.columnas), dtype=np.float32)
        fila_visible = min(self.fila_pelota, self.filas - 1)
        grid[fila_visible, self.columna_pelota] = 1.0
        grid[self.filas - 1, self.columna_pala:self.columna_pala + self.ancho_pala] = 1.0
        return grid.flatten()

    def step(self, accion):
        desplazamiento = {0: -1, 1: 0, 2: 1}[accion]
        self.columna_pala = int(
            np.clip(self.columna_pala + desplazamiento, 0, self.columnas - self.ancho_pala)
        )
        self.fila_pelota += self.velocidad_caida
        terminado = self.fila_pelota >= self.filas - 1
        if terminado:
            atrapada = (
                self.columna_pala
                <= self.columna_pelota
                < self.columna_pala + self.ancho_pala
            )
            recompensa = 1.0 if atrapada else -1.0
        else:
            recompensa = 0.0
        return self._estado(), recompensa, terminado


# --- Parte B: ciclo sperimentale ---
N_EPISODIOS = 2000

EXPERIMENTOS = {
    "base (10x5, pala=2, vel=1)": dict(filas=10, columnas=5, ancho_pala=2, velocidad_caida=1),
    "mas columnas (10x8, pala=2, vel=1)": dict(filas=10, columnas=8, ancho_pala=2, velocidad_caida=1),
    "pala mas estrecha (10x5, pala=1, vel=1)": dict(filas=10, columnas=5, ancho_pala=1, velocidad_caida=1),
}

resultados = {}
for nombre, config in EXPERIMENTOS.items():
    entorno = CatchParametrizable(**config)
    aciertos = entrenar_agente(entorno, N_EPISODIOS)
    ep_umbral = episodios_hasta_umbral(aciertos)
    resultados[nombre] = (aciertos, ep_umbral)
```

Con queste tre configurazioni vedrai che la base (dieci righe, cinque colonne, racchetta di due pixel) raggiunge l'80% di successi in qualche centinaio di episodi, mentre allargare la griglia a otto colonne o ridurre la racchetta a un solo pixel costringe l'agente a impiegare parecchio di più: ci sono più colonne possibili in cui può cadere la palla, oppure meno margine di errore per coprirla, quindi la politica ha bisogno di più esperienza per affinare la mira.
</details>

## Per continuare a esercitarti

**Porta `velocidad_caida` a 3 o 4 in una nuova configurazione e riesegui l'esperimento.** Impiega più o meno tempo della configurazione base per arrivare all'80% di successi? Rifletti sul perché: con una palla che cade più in fretta, l'agente ha meno passi per episodio per correggere la posizione della racchetta, ma anche ogni episodio è più corto e più economico da addestrare.

Annota con quale configurazione esatta il tuo agente comincia a fallire in 2.000 episodi, e quale modifica —più episodi, più entropia, una rete più grande— lo sistema di nuovo. Questo esercizio di sondare i limiti di un problema piccolo è, parola per parola, il consiglio pratico centrale della lezione 11.